# 11 — Validated SAE controls and feature-activity diagnostics (repository runner)

This is a repository-native companion to the detailed notebook 11 distributed with the research review. The common causal v4 engine performs exact temporal residual patching, active dose-matched nonsemantic control, spatial signflip, spatial permutation, independent discovery and target/replacement/context scoring. It writes into a separate v4 repository run folder.

In [ ]:
MODE="pilot"  # pilot or full
EVALUATION_SEEDS=(0,) if MODE=="pilot" else (1,2)
MAX_PAIRS_PER_LABEL=1 if MODE=="pilot" else None
FORCE_RERUN=False
ACTIVITY_SAMPLE_TOKENS=256

In [ ]:
import json,os,sys
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
def root():
    wd=Path.cwd().resolve()
    for p in [wd,*wd.parents,Path("/home/jupyter/project/diffusion_attention_analysis_2")]:
        if (p/"src/diffusion_attention_analysis_v2").is_dir():return p
    raise FileNotFoundError("Project root unavailable")
ROOT=root();os.chdir(ROOT);sys.path.insert(0,str(ROOT/"src"))
from diffusion_attention_analysis_v2.config import load_config,resolve_path
from diffusion_attention_analysis_v2.sd3_notebook_utils import prepare_datashere_storage
from diffusion_attention_analysis_v2.sd3_counterfactual_v4 import run_campaign,tensor_ref
from diffusion_attention_analysis_v2.sae_fidelity_v4 import load_research_sae,encode_at
storage=prepare_datashere_storage(project_root=ROOT,run_tag="sd3_medium_sae_circuit_150",container_name="container4",require_container=True,min_free_gb=12.)
assert storage["free_space_ok"] and torch.cuda.is_available()
cfg=load_config(ROOT/"configs/06_causal_analysis/sd3_medium_counterfactual_sae_mediation_v2.yaml")
PREV=resolve_path(cfg,"data.output_dir",required=True)
manifest=pd.read_csv(PREV/"split_manifest.csv")
RUN=ROOT/"outputs/sd3_medium_sae_circuit_150"
OUT=RUN/"06_causal_completeness_v4_repository"
CHECKPOINT=ROOT/"outputs/sd3_medium_sae_circuit_150/03_sae/mid_block11_step13/sae.pt"
assert CHECKPOINT.is_file()
ROOTS=[PREV,RUN/"06_causal_completeness_v3",RUN/"06_causal_completeness_v4"]
print("Previous discovery",PREV,"New v4 artifacts",OUT)

## 1. Causal controls, Grounding DINO and downstream probe

A control with zero applied intervention is a failure, not a negative result. Discovery never uses the evaluation prompts.

In [ ]:
summary,primary=run_campaign(cfg,manifest,output=OUT,reference_roots=ROOTS,
    checkpoint=CHECKPOINT,discovery_seed=0,evaluation_seeds=EVALUATION_SEEDS,
    max_eval_pairs=MAX_PAIRS_PER_LABEL,force=FORCE_RERUN)
display(summary)
display(pd.DataFrame([primary]))
telemetry=pd.read_csv(OUT/"intervention_telemetry.csv")
assert telemetry.loc[telemetry.arm.eq("active_control_16"),"mean_dose"].min()>0
assert telemetry.loc[telemetry.arm.eq("signflip_16"),"mean_dose"].min()>0
display(telemetry.groupby("arm").agg(mean_dose=("mean_dose","mean"),downstream=("probe_recovery","mean")))

## 2. Activity, dead coordinates and top-K latent-mass coverage

This is a proxy for SAE representation activity, not an additive causal share. It diagnoses K-saturation before subsequent checkpoint comparisons.

In [ ]:
import numpy as np
from diffusion_attention_analysis_v2.causal_controls import weighted_latent_coverage
model,_=load_research_sae(CHECKPOINT,"cuda")
rank=json.loads((OUT/"frozen_feature_ranks.json").read_text())
effects=pd.read_csv(OUT/"paired_effects.csv")
selected=manifest[manifest.split.isin(["evaluation","confirmatory"])]
if MAX_PAIRS_PER_LABEL is not None:selected=selected.groupby("label",group_keys=False).head(MAX_PAIRS_PER_LABEL)
rows=[]
for _,p in selected.iterrows():
    for seed in EVALUATION_SEEDS:
        source=tensor_ref([OUT,*ROOTS],p.pair_id,seed,"source","transformer_blocks.11",13)
        destination=tensor_ref([OUT,*ROOTS],p.pair_id,seed,"destination","transformer_blocks.11",13)
        rg=torch.Generator().manual_seed(2601+int(seed))
        sel=torch.randperm(len(source),generator=rg)[:ACTIVITY_SAMPLE_TOKENS]
        with torch.no_grad():
            zs=encode_at(model,source[sel].to("cuda"),13).float().cpu()
            zd=encode_at(model,destination[sel].to("cuda"),13).float().cpu()
        d=zs-zd
        row={"pair_id":p.pair_id,"label":p.label,"seed":int(seed),
             "changed_latents":int((d.abs().amax(0)>1e-7).sum()),
             "nonzero_latents_source":int((zs.abs().sum(0)>1e-7).sum())}
        row.update({f"coverage_{k}":v for k,v in weighted_latent_coverage(
            zs,zd,model.decoder.weight.detach().cpu(),rank[str(p.label)],[4,16,32,64,128,256,512]).items()})
        rows.append(row)
activity=pd.DataFrame(rows)
activity.to_csv(OUT/"feature_activity_coverage.csv",index=False)
display(activity.groupby("label").mean(numeric_only=True))
print("Mean coverage increment from K32 to K512:",float((activity.coverage_512-activity.coverage_32).mean()))
del model;torch.cuda.empty_cache()